# Проверка чтения Google Sheets

Этот ноутбук читает указанный диапазон ячеек из Google Sheets в `pandas.DataFrame` и показывает результат.

1. Укажи путь к JSON сервисного аккаунта
2. Укажи `spreadsheet_id`, имя листа и диапазон
3. Запусти ячейки сверху вниз


In [7]:
from pathlib import Path

# Заполни эти значения под свою таблицу
SERVICE_ACCOUNT_JSON = Path("C:\\vscodeproj\\tg_bot_financial\\credentials\\gen-lang-client-0657346511-92aecc7038e3.json")
SPREADSHEET_ID = "11s-9cDXTJSiuXljTaxwEDwdiSiUu_GwKP2l_qCJeKeM"
WORKSHEET_NAME = "Транзакции"
CELL_RANGE = "B4:E200"
RANGE_HAS_HEADER = True


In [8]:
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

if not SERVICE_ACCOUNT_JSON.exists():
    raise FileNotFoundError(f"Не найден файл сервисного аккаунта: {SERVICE_ACCOUNT_JSON}")

if "<YOUR_GOOGLE_SPREADSHEET_ID>" in SPREADSHEET_ID:
    raise ValueError("Замени SPREADSHEET_ID на реальный идентификатор таблицы.")

credentials = Credentials.from_service_account_file(
    str(SERVICE_ACCOUNT_JSON),
    scopes=SCOPES,
)
client = gspread.authorize(credentials)
spreadsheet = client.open_by_key(SPREADSHEET_ID)
worksheet = spreadsheet.worksheet(WORKSHEET_NAME)
values = worksheet.get(CELL_RANGE)

if not values:
    df = pd.DataFrame()
else:
    max_len = max(len(row) for row in values)
    normalized = [row + [None] * (max_len - len(row)) for row in values]

    if RANGE_HAS_HEADER:
        header = normalized[0]
        rows = normalized[1:]
        df = pd.DataFrame(rows, columns=header)
    else:
        df = pd.DataFrame(normalized)

print(f"Лист: {WORKSHEET_NAME}")
print(f"Диапазон: {CELL_RANGE}")
print(f"Строк в DataFrame: {len(df)}")


Лист: Транзакции
Диапазон: B4:E200
Строк в DataFrame: 108


In [10]:
e4_value = worksheet.acell("E4").value
categories_from_e4 = [] if not e4_value else [item.strip() for item in e4_value.split(",") if item.strip()]
print(f"???????? ?????? E4: {e4_value}")
categories_from_e4


???????? ?????? E4: Категория


['Категория']

In [13]:
e5 = worksheet.acell("E5")
e5

<Cell R5C5 'Вкусности'>

In [9]:
df


,Дата,Сумма,Описание,Категория
0,01.03.2026,"175,00 ₽",самокат,Вкусности
1,01.03.2026,"48093,00 ₽",ПК,Гаджеты
2,01.03.2026,"294,00 ₽",самокат,Продукты базовые
3,01.03.2026,"400,00 ₽",Сберпрайм,Обязательные платежи/Подписки
4,01.03.2026,"1040,00 ₽",рамен,Общепит/Доставка
...,...,...,...,...
103,30.03.2026,"720,00 ₽",Лента,Дом
104,30.03.2026,"230,00 ₽",Лента,Продукты базовые
105,30.03.2026,"325,00 ₽",Лента,Вкусности
106,31.03.2026,"69,00 ₽",Колос,Вкусности


In [14]:
from google.oauth2.service_account import Credentials
from google.auth.transport.requests import AuthorizedSession

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

creds = Credentials.from_service_account_file(
    str(SERVICE_ACCOUNT_JSON),
    scopes=SCOPES,
)
session = AuthorizedSession(creds)

target_range = f"{WORKSHEET_NAME}!E5:E200"
url = f"https://sheets.googleapis.com/v4/spreadsheets/{SPREADSHEET_ID}"
params = {
    "ranges": target_range,
    "includeGridData": "true",
    "fields": "sheets(data(rowData(values(dataValidation))))",
}

resp = session.get(url, params=params)
resp.raise_for_status()
payload = resp.json()

dropdown_info = []
rows = payload["sheets"][0]["data"][0].get("rowData", [])

for row_index, row in enumerate(rows, start=5):
    values = row.get("values", [])
    cell = values[0] if values else {}
    dv = cell.get("dataValidation")
    if not dv:
        continue

    condition = dv.get("condition", {})
    condition_type = condition.get("type")
    condition_values = [
        item.get("userEnteredValue")
        for item in condition.get("values", [])
        if item.get("userEnteredValue") is not None
    ]

    dropdown_info.append({
        "row": row_index,
        "type": condition_type,
        "values": condition_values,
    })

dropdown_info[:5]


[{'row': 135, 'type': 'ONE_OF_RANGE', 'values': ["='Сводка'!$B$27:$C$50"]},
 {'row': 136, 'type': 'ONE_OF_RANGE', 'values': ["='Сводка'!$B$27:$C$50"]},
 {'row': 137, 'type': 'ONE_OF_RANGE', 'values': ["='Сводка'!$B$27:$C$50"]},
 {'row': 138, 'type': 'ONE_OF_RANGE', 'values': ["='Сводка'!$B$27:$C$45"]},
 {'row': 139, 'type': 'ONE_OF_RANGE', 'values': ["='Сводка'!$B$27:$C$50"]}]

In [16]:
CATEGORY_SOURCE_RANGE = "'Сводка'!B27:C50"

values = spreadsheet.values_get(CATEGORY_SOURCE_RANGE).get("values", [])
categories = [value.strip() for row in values for value in row if value and value.strip()]
categories = list(dict.fromkeys(categories))

print(f"Source range: {CATEGORY_SOURCE_RANGE}")
print(f"Categories count: {len(categories)}")
categories


Source range: 'Сводка'!B27:C50
Categories count: 20


['Общепит/Доставка',
 'Задолженности',
 'Одежда',
 'Здоровье/Медицина/Уход',
 'Продукты базовые',
 'Вкусности',
 'Другое',
 'Обед',
 'Обязательные платежи/Подписки',
 'Досуг',
 'Подарки',
 'Дом',
 'Транспорт',
 'Домашние животные',
 'Личные расходы',
 'Инвестиции',
 'Дал в долг',
 'Путешествия',
 'Зарплата жены',
 'Гаджеты']